# OPALX vs Direct LW Picard Comparison

This notebook compares OPALX `.stat` moments against the direct Lienard-Wiechert Picard solver in `run_lw_picard.py`. The plotted quantities match `opalx-comparison.ipynb` and `opalx-tps-comparison-fair.ipynb`: `rms_x`, `rms_y`, `rms_s`, `rms_px`, `rms_py`, `rms_ps`, `energy`, and `dE`.

The Picard solver is not a PIC/grid solver. Iteration 0 is the external-field-only analytic trajectory. Iterations 1..3 repeatedly evaluate direct retarded Lienard-Wiechert pair fields from the previous full trajectory history. The default run uses Gaussian initial sampling, 512 particles, and 801 stored points, so the 800 intervals match `opalx-sim/pert-test-uniformsphere.stat`.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "python" / "lw_perturbation" / "run_lw_picard.py").exists():
            return path
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
LW_DIR = REPO_ROOT / "python" / "lw_perturbation"
PICARD_SCRIPT = LW_DIR / "run_lw_picard.py"
OPALX_STAT = LW_DIR / "opalx-sim" / "pert-test-uniformsphere.stat"
PICARD_OUTPUT = LW_DIR / "output_picard_gaussian"
PICARD_MOMENTS = PICARD_OUTPUT / "picard_moments_by_iteration.csv"
PLOT_PATH = LW_DIR / "comparison-picard-gaussian-pert-test-uniformsphere-iter1-3-N512-800steps.png"

PICARD_RUN_COMMAND = [
    sys.executable,
    str(PICARD_SCRIPT),
    "--particles", "512",
    "--outputs", "801",
    "--iterations", "3",
    "--t-end", "3e-9",
    "--initial-energy-gev", "1e-9",
    "--retarded-iterations", "3",
    "--distribution", "gaussian",
    "--output-dir", str(PICARD_OUTPUT),
    "--match-initial-stat", str(OPALX_STAT),
    "--no-save-all-iterations",
]

In [ ]:
def read_sdds_file(filename):
    """Read an OPAL/OPALX SDDS-format .stat/.dat file into a DataFrame."""
    col_names = []
    col_info = {}
    header_lines = 0
    first_data_line = None

    with open(filename, "r") as f:
        for line in f:
            header_lines += 1
            stripped = line.strip()

            if stripped.startswith("&column"):
                col_def = stripped
                while "&end" not in col_def:
                    next_line = next(f).strip()
                    header_lines += 1
                    col_def += " " + next_line

                name_match = re.search(r"name\s*=\s*(\S+)", col_def)
                type_match = re.search(r"type\s*=\s*(\S+)", col_def)
                unit_match = re.search(r"units\s*=\s*(\S+)", col_def)
                desc_match = re.search(r'description\s*=\s*"([^"]*)"', col_def)

                name = name_match.group(1).strip(",\"") if name_match else f"col_{len(col_names)}"
                dtype = type_match.group(1).strip(",\"") if type_match else "double"
                unit = unit_match.group(1).strip(",\"") if unit_match else ""
                desc = desc_match.group(1) if desc_match else ""

                col_names.append(name)
                col_info[name] = (dtype, unit, desc)

            if stripped.startswith("&data"):
                while True:
                    next_line = next(f).strip()
                    header_lines += 1
                    parts = next_line.split()
                    if not parts:
                        continue
                    try:
                        values = [float(part) for part in parts]
                    except ValueError:
                        continue
                    if len(values) == len(col_names):
                        first_data_line = next_line
                        break
                break

    if first_data_line is None:
        raise RuntimeError(f"Could not find numeric data in SDDS file: {filename}")

    data_rows = [[float(x) for x in first_data_line.split()]]
    with open(filename, "r") as f:
        for _ in range(header_lines):
            next(f)
        for line in f:
            stripped = line.strip()
            if stripped and not stripped.startswith("&"):
                try:
                    values = [float(x) for x in stripped.split()]
                    if len(values) == len(col_names):
                        data_rows.append(values)
                except ValueError:
                    continue

    return pd.DataFrame(data_rows, columns=col_names), col_info


def ensure_picard_results(force=False):
    """Run `run_lw_picard.py` when the cached moments file is missing."""
    if PICARD_MOMENTS.exists() and not force:
        return PICARD_MOMENTS
    PICARD_OUTPUT.mkdir(parents=True, exist_ok=True)
    print("running", " ".join(PICARD_RUN_COMMAND))
    subprocess.run(PICARD_RUN_COMMAND, cwd=REPO_ROOT, check=True)
    return PICARD_MOMENTS


def load_picard_moments(moments_path=PICARD_MOMENTS):
    moments = pd.read_csv(moments_path)
    return pd.DataFrame({
        "iteration": moments["iteration"].astype(int),
        "t": moments["time_ns"],
        "rms_x": moments["rms_x_m"],
        "rms_y": moments["rms_y_m"],
        "rms_s": moments["rms_z_m"],
        "rms_px": moments["rms_px_beta_gamma"],
        "rms_py": moments["rms_py_beta_gamma"],
        "rms_ps": moments["rms_pz_beta_gamma"],
        "energy": moments["mean_kinetic_energy_MeV"],
        "dE": moments["energy_spread_MeV"],
    })

In [ ]:
def compare_opalx_to_picard(
    stat_path=OPALX_STAT,
    moments_path=PICARD_MOMENTS,
    iterations=(1, 2, 3),
    columns=None,
    data_range=None,
    save_as=PLOT_PATH,
    ncols=2,
    force_picard=False,
):
    """Compare OPALX .stat output against direct LW Picard iterations."""
    if columns is None:
        columns = [
            "rms_x",
            "rms_y",
            "rms_s",
            "rms_px",
            "rms_py",
            "rms_ps",
            "energy",
            "dE",
        ]
    columns = list(columns)

    ensure_picard_results(force=force_picard)
    opalx, _ = read_sdds_file(stat_path)
    picard = load_picard_moments(moments_path)

    if iterations is None:
        iterations = sorted(picard["iteration"].unique())
    else:
        iterations = [int(value) for value in iterations]
    missing_iterations = sorted(set(iterations) - set(picard["iteration"].unique()))
    if missing_iterations:
        raise ValueError(f"Missing Picard iterations in {moments_path}: {missing_iterations}")

    missing_opalx = [col for col in columns if col not in opalx.columns]
    missing_picard = [col for col in columns if col not in picard.columns]
    if missing_opalx:
        raise ValueError(f"Missing OPALX columns: {missing_opalx}")
    if missing_picard:
        raise ValueError(f"Missing Picard columns: {missing_picard}")

    if data_range is None:
        row_slice = slice(None)
    else:
        start, stop = data_range
        row_slice = slice(start, stop)

    opalx_view = opalx.iloc[row_slice].copy()
    opalx_t = opalx_view["t"].to_numpy()
    picard_t = picard.loc[picard["iteration"] == iterations[0], "t"].to_numpy()
    valid_time = (opalx_t >= picard_t.min()) & (opalx_t <= picard_t.max())
    if not np.any(valid_time):
        raise ValueError("OPALX and Picard time grids do not overlap")
    opalx_view = opalx_view.loc[valid_time].copy()
    opalx_t = opalx_view["t"].to_numpy()

    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 150,
        "savefig.dpi": 200,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "grid.linestyle": "--",
    })

    labels = {
        "rms_x": "RMS x [m]",
        "rms_y": "RMS y [m]",
        "rms_s": "RMS z/s [m]",
        "rms_px": "RMS px [beta*gamma]",
        "rms_py": "RMS py [beta*gamma]",
        "rms_ps": "RMS pz/ps [beta*gamma]",
        "energy": "mean kinetic energy [MeV]",
        "dE": "energy spread [MeV]",
    }

    ncols = max(1, min(int(ncols), len(columns)))
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(7 * ncols, 2.6 * nrows),
        sharex=True,
        constrained_layout=True,
        squeeze=False,
    )

    summary_rows = []
    for ax, col in zip(axes.ravel(), columns):
        opalx_y = opalx_view[col].to_numpy()
        ax.plot(opalx_t, opalx_y, label="OPALX .stat", lw=1.8, color="black")
        err_for_axis = []

        for iteration in iterations:
            curve = picard[picard["iteration"] == iteration].sort_values("t")
            curve_t = curve["t"].to_numpy()
            curve_y = np.interp(opalx_t, curve_t, curve[col].to_numpy())
            abs_err = np.abs(opalx_y - curve_y)
            err_for_axis.append(abs_err)
            ax.plot(opalx_t, curve_y, label=f"LW Picard iter={iteration}", lw=1.6)

            denom = np.maximum(np.abs(curve_y), 1e-30)
            summary_rows.append({
                "iteration": iteration,
                "column": col,
                "max_abs_error": np.max(abs_err),
                "mean_abs_error": np.mean(abs_err),
                "max_relative_error": np.max(abs_err / denom),
                "mean_relative_error": np.mean(abs_err / denom),
            })

        ax.set_title(col)
        ax.set_ylabel(labels.get(col, col))
        ax.legend(loc="best", framealpha=0.7)

        panel_err = np.min(np.vstack(err_for_axis), axis=0)
        ax_err = ax.twinx()
        ax_err.plot(opalx_t, panel_err, color="silver", lw=1.2, ls="--", alpha=0.8, label="min |Delta|")
        ax_err.set_ylabel("min |Delta|", color="silver", fontsize=9)
        ax_err.tick_params(axis="y", colors="silver", labelsize=8)
        ax_err.yaxis.get_offset_text().set_color("silver")
        if np.any(panel_err > 0):
            ax_err.set_yscale("log")

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)
    for ax in axes[-1, :]:
        if ax.get_visible():
            ax.set_xlabel("t [ns]")

    if save_as is not None:
        save_as = Path(save_as)
        save_as.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_as)
        print(f"saved {save_as}")
    plt.show()

    return pd.DataFrame(summary_rows)

In [ ]:
comparison_summary = compare_opalx_to_picard(
    stat_path=OPALX_STAT,
    moments_path=PICARD_MOMENTS,
    iterations=(1, 2, 3),
    save_as=PLOT_PATH,
    # force_picard=True,  # uncomment to rerun run_lw_picard.py
)
comparison_summary